In [2]:

# ============================================================
# PROYECTO FINAL - BIOSEÑALES Y SISTEMAS
# SELECCIÓN DE UNA MUESTRA DE ECG PARA EL PROYECTO 2
# ============================================================

import os
import zipfile
import random
import pandas as pd
from pathlib import PurePosixPath
from collections import defaultdict

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

RUTA_DIAGNOSTICOS = "Diagnostics.xlsx"
CARPETA_SALIDA = "ECGData_muestra"

ARCHIVO_SALIDA = "ECGData_muestra_2500.zip"
ARCHIVO_TABLA = "Diagnostics_muestra_2500.csv"

EDAD_MINIMA = 18
N_POR_GRUPO = 500
SEMILLA = 42

# ------------------------------------------------------------
# 2. DEFINICIÓN DE LOS GRUPOS
# ------------------------------------------------------------

GRUPO_RITMO = {
    "SB": "SB",

    "AFIB": "AFIB",
    "AF": "AFIB",

    "SVT": "GSVT",
    "AT": "GSVT",
    "AVNRT": "GSVT",
    "AVRT": "GSVT",
    "SAAWR": "GSVT",

    "SR": "SR",
    "SA": "SR",

    "ST": "ST"
}

ORDEN_GRUPOS = ["SB", "AFIB", "SR", "ST", "GSVT"]

# ------------------------------------------------------------
# 3. VERIFICAR ARCHIVOS
# ------------------------------------------------------------

print("=" * 65)
print("VERIFICACIÓN DE ARCHIVOS")
print("=" * 65)

if not os.path.exists(RUTA_DIAGNOSTICOS):
    raise FileNotFoundError(
        f"No se encontró {RUTA_DIAGNOSTICOS}. "
        "Revisa la ubicación del archivo."
    )

if not os.path.exists(RUTA_ECG):
    raise FileNotFoundError(
        f"No se encontró {RUTA_ECG}. "
        "Revisa la ubicación del archivo."
    )

print("Diagnostics.xlsx encontrado")
print("ECGData.zip encontrado")

print(
    "Tamaño del archivo original:",
    round(os.path.getsize(RUTA_ECG) / (1024**2), 2),
    "MB"
)

# ------------------------------------------------------------
# 4. CARGAR INFORMACIÓN CLÍNICA
# ------------------------------------------------------------

diag = pd.read_excel(RUTA_DIAGNOSTICOS)

print("\nDimensiones originales:", diag.shape)
print("\nColumnas disponibles:")
print(diag.columns.tolist())

# Normalizar nombres de registros
diag["FileName"] = diag["FileName"].astype(str).str.strip()

# Asignar grupo a cada registro
diag["Grupo"] = diag["Rhythm"].map(GRUPO_RITMO)

# Verificar que todos los ritmos tengan grupo
sin_grupo = diag.loc[diag["Grupo"].isna(), "Rhythm"].unique()

if len(sin_grupo) > 0:
    raise ValueError(
        f"Hay ritmos sin clasificación: {sin_grupo}"
    )

# ------------------------------------------------------------
# 5. FILTRAR PACIENTES ADULTOS
# ------------------------------------------------------------

diag["PatientAge"] = pd.to_numeric(
    diag["PatientAge"], errors="coerce"
)

diag_adultos = diag[
    diag["PatientAge"] >= EDAD_MINIMA
].copy()

print("\nRegistros de pacientes adultos:",
      len(diag_adultos))

# ------------------------------------------------------------
# 6. ANALIZAR DISPONIBILIDAD POR GRUPO
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("DISPONIBILIDAD DE REGISTROS POR GRUPO")
print("=" * 65)

disponibilidad = (
    diag_adultos["Grupo"]
    .value_counts()
    .reindex(ORDEN_GRUPOS, fill_value=0)
)

print(disponibilidad.to_string())

# Comprobar que hay suficientes registros
insuficientes = disponibilidad[
    disponibilidad < N_POR_GRUPO
]

if len(insuficientes) > 0:
    raise ValueError(
        "No hay suficientes registros para seleccionar "
        f"{N_POR_GRUPO} por grupo:\n{insuficientes}"
    )

# ------------------------------------------------------------
# 7. SELECCIÓN ALEATORIA ESTRATIFICADA
# ------------------------------------------------------------

muestra = (
    diag_adultos
    .groupby("Grupo", group_keys=False)
    .apply(
        lambda grupo: grupo.sample(
            n=N_POR_GRUPO,
            random_state=SEMILLA
        ),
        include_groups=True
    )
    .reset_index(drop=True)
)

# Organizar los registros por grupo
muestra["Grupo"] = pd.Categorical(
    muestra["Grupo"],
    categories=ORDEN_GRUPOS,
    ordered=True
)

muestra = muestra.sort_values(
    ["Grupo", "FileName"]
).reset_index(drop=True)

print("\n" + "=" * 65)
print("MUESTRA SELECCIONADA")
print("=" * 65)

print(
    muestra["Grupo"]
    .value_counts()
    .reindex(ORDEN_GRUPOS)
    .to_string()
)

print("\nTotal de registros seleccionados:", len(muestra))

# ------------------------------------------------------------

# ============================================================
# 8. EXTRAER LOS ECG SELECCIONADOS EN UNA CARPETA
# ============================================================

os.makedirs(CARPETA_SALIDA, exist_ok=True)

indice_archivos = defaultdict(list)

with zipfile.ZipFile(RUTA_ECG, "r") as zip_original:

    archivos_csv = [
        nombre for nombre in zip_original.namelist()
        if nombre.lower().endswith(".csv")
    ]

    for ruta in archivos_csv:
        nombre_base = PurePosixPath(ruta).stem
        indice_archivos[nombre_base].append(ruta)

    encontrados = 0
    no_encontrados = []

    for _, registro in muestra.iterrows():

        nombre = str(registro["FileName"]).strip()

        # Evitar duplicar la extensión .csv
        nombre_base = os.path.splitext(nombre)[0]

        coincidencias = indice_archivos.get(nombre_base, [])

        if len(coincidencias) != 1:
            no_encontrados.append(nombre)
            continue

        ruta_original = coincidencias[0]

        contenido = zip_original.read(ruta_original)

        ruta_destino = os.path.join(
            CARPETA_SALIDA,
            nombre_base + ".csv"
        )

        with open(ruta_destino, "wb") as archivo:
            archivo.write(contenido)

        encontrados += 1

# ============================================================
# 9. GUARDAR LA INFORMACIÓN CLÍNICA
# ============================================================

muestra.to_csv(
    "Diagnostics_muestra_2500.csv",
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 10. INFORME FINAL
# ============================================================

print("=" * 60)
print("RESULTADOS DE LA SELECCIÓN")
print("=" * 60)

print("Registros seleccionados:", len(muestra))
print("ECG guardados:", encontrados)
print("ECG no encontrados:", len(no_encontrados))

if no_encontrados:
    print("Registros pendientes:")
    print(no_encontrados[:20])

print("\nArchivos generados:")
print("Carpeta:", CARPETA_SALIDA)
print("Tabla: Diagnostics_muestra_2500.csv")

print("\nPROCESO FINALIZADO")


# ------------------------------------------------------------
# 11. INFORME FINAL
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("RESULTADOS DEL PROCESO")
print("=" * 65)

print("Registros seleccionados:", len(muestra))
print("ECG encontrados:", encontrados)
print("ECG no encontrados:", len(no_encontrados))
print("Nombres duplicados en el ZIP:", len(duplicados_nombre))

if no_encontrados:
    print("\nRegistros sin archivo correspondiente:")
    print(no_encontrados[:20])

tamano_final = os.path.getsize(ARCHIVO_SALIDA) / (1024**2)

print("\nTamaño del nuevo ZIP:",
      round(tamano_final, 2), "MB")

print("\nArchivo ZIP:", ARCHIVO_SALIDA)
print("Tabla clínica:", ARCHIVO_TABLA)

# ------------------------------------------------------------
# 12. VERIFICACIÓN DE LA MUESTRA
# ------------------------------------------------------------

print("\nDistribución definitiva:")

resumen = (
    muestra.groupby("Grupo", observed=False)
    .agg(
        Registros=("FileName", "count"),
        Edad_promedio=("PatientAge", "mean"),
        Edad_minima=("PatientAge", "min"),
        Edad_maxima=("PatientAge", "max"),
        Mujeres=("Gender", lambda x: (x == "FEMALE").sum()),
        Hombres=("Gender", lambda x: (x == "MALE").sum())
    )
)

print(resumen.round(2))

# Verificar integridad del nuevo ZIP
with zipfile.ZipFile(ARCHIVO_SALIDA, "r") as z:
    error = z.testzip()

if error is None:
    print("\nVERIFICACIÓN EXITOSA: ZIP sin errores de integridad.")
else:
    print("\nERROR en el archivo:", error)

print("\nPROCESO FINALIZADO.")


VERIFICACIÓN DE ARCHIVOS
Diagnostics.xlsx encontrado
ECGData.zip encontrado
Tamaño del archivo original: 719.62 MB

Dimensiones originales: (10646, 16)

Columnas disponibles:
['FileName', 'Rhythm', 'Beat', 'PatientAge', 'Gender', 'VentricularRate', 'AtrialRate', 'QRSDuration', 'QTInterval', 'QTCorrected', 'RAxis', 'TAxis', 'QRSCount', 'QOnset', 'QOffset', 'TOffset']

Registros de pacientes adultos: 10381

DISPONIBILIDAD DE REGISTROS POR GRUPO
Grupo
SB      3871
AFIB    2225
SR      2081
ST      1483
GSVT     721

MUESTRA SELECCIONADA
Grupo
SB      500
AFIB    500
SR      500
ST      500
GSVT    500

Total de registros seleccionados: 2500


C:\Users\giral\AppData\Local\Temp\ipykernel_18168\3360148176.py:153: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


RESULTADOS DE LA SELECCIÓN
Registros seleccionados: 2500
ECG guardados: 2500
ECG no encontrados: 0

Archivos generados:
Carpeta: ECGData_muestra
Tabla: Diagnostics_muestra_2500.csv

PROCESO FINALIZADO

RESULTADOS DEL PROCESO
Registros seleccionados: 2500
ECG encontrados: 2500
ECG no encontrados: 0
Nombres duplicados en el ZIP: 0

Tamaño del nuevo ZIP: 194.24 MB

Archivo ZIP: ECGData_muestra_2500.zip
Tabla clínica: Diagnostics_muestra_2500.csv

Distribución definitiva:
       Registros  Edad_promedio  Edad_minima  Edad_maxima  Mujeres  Hombres
Grupo                                                                      
SB           500          59.43           20           92      186      314
AFIB         500          72.85           26           94      211      289
SR           500          53.31           18           88      295      205
ST           500          57.23           18           96      240      260
GSVT         500          58.38           20           96      256     

In [3]:

# ============================================================
# PROYECTO 2 - VERIFICACIÓN DE LOS ARCHIVOS ECG
# ============================================================

import os
import glob
import pandas as pd
import numpy as np

# Rutas de trabajo
CARPETA_ECG = "ECGData_muestra"
RUTA_DIAGNOSTICOS = "Diagnostics_muestra_2500.csv"

# 1. Verificar que existan las carpetas y archivos
assert os.path.isdir(CARPETA_ECG), (
    f"No se encuentra la carpeta {CARPETA_ECG}"
)

assert os.path.isfile(RUTA_DIAGNOSTICOS), (
    f"No se encuentra {RUTA_DIAGNOSTICOS}"
)

# 2. Identificar los archivos ECG
archivos_ecg = sorted(
    glob.glob(os.path.join(CARPETA_ECG, "*.csv"))
)

# 3. Cargar información clínica
diagnosticos = pd.read_csv(RUTA_DIAGNOSTICOS)

print("=" * 60)
print("VERIFICACIÓN DE LA BASE DE DATOS")
print("=" * 60)

print("Número de señales ECG:", len(archivos_ecg))
print("Número de registros clínicos:", len(diagnosticos))

print("\nColumnas de información clínica:")
print(diagnosticos.columns.tolist())

print("\nDistribución de las clases:")
print(diagnosticos["Grupo"].value_counts())

# 4. Comprobar correspondencia entre señales y diagnósticos
nombres_ecg = {
    os.path.splitext(os.path.basename(f))[0]
    for f in archivos_ecg
}

nombres_diag = set(
    diagnosticos["FileName"].astype(str)
    .str.replace(r"\.csv$", "", regex=True)
)

sin_senal = nombres_diag - nombres_ecg
sin_diagnostico = nombres_ecg - nombres_diag

print("\nRegistros sin señal:", len(sin_senal))
print("Señales sin diagnóstico:", len(sin_diagnostico))

# 5. Inspeccionar un archivo ECG
if archivos_ecg:

    archivo_prueba = archivos_ecg[0]

    print("\n" + "=" * 60)
    print("INSPECCIÓN DE UNA SEÑAL ECG")
    print("=" * 60)

    print("Archivo:", os.path.basename(archivo_prueba))

    senal_prueba = pd.read_csv(
        archivo_prueba,
        header=None
    )

    print("\nDimensiones de la señal:")
    print(senal_prueba.shape)

    print("\nPrimeras cinco filas:")
    print(senal_prueba.head())

    print("\nÚltimas cinco filas:")
    print(senal_prueba.tail())

    print("\nTipos de datos:")
    print(senal_prueba.dtypes)

    print("\nValores faltantes:")
    print(senal_prueba.isna().sum().sum())

    print("\nResumen estadístico:")
    print(senal_prueba.describe())

else:
    print("No se encontraron archivos ECG.")


VERIFICACIÓN DE LA BASE DE DATOS
Número de señales ECG: 2500
Número de registros clínicos: 2500

Columnas de información clínica:
['FileName', 'Rhythm', 'Beat', 'PatientAge', 'Gender', 'VentricularRate', 'AtrialRate', 'QRSDuration', 'QTInterval', 'QTCorrected', 'RAxis', 'TAxis', 'QRSCount', 'QOnset', 'QOffset', 'TOffset', 'Grupo']

Distribución de las clases:
Grupo
SB      500
AFIB    500
SR      500
ST      500
GSVT    500
Name: count, dtype: int64

Registros sin señal: 0
Señales sin diagnóstico: 0

INSPECCIÓN DE UNA SEÑAL ECG
Archivo: MUSE_20180111_155154_74000.csv

Dimensiones de la señal:
(5001, 12)

Primeras cinco filas:
       0       1       2      3      4       5       6      7      8       9   \
0       I      II     III    aVR    aVL     aVF      V1     V2     V3      V4   
1  -14.64   -24.4   -9.76  19.52  -4.88  -19.52   -24.4   24.4  34.16       0   
2  -14.64  -43.92  -29.28  29.28   4.88  -39.04  -29.28  34.16  58.56   -4.88   
3   -9.76   -24.4  -14.64  19.52      0  -